In [1]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    !pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Ambiente Locale rilevato. Procedo con l'esecuzione...
✅ Collegamento ai dati riuscito! Cartella raw: d:\GitHub repositories\crop-spatial-classification\data\raw


In [ ]:
import os
import glob # Libreria per cercare file nelle sottocartelle
import pandas as pd
import rasterio
import numpy as np

ground_truth = pd.read_json(f"{DATA_DIR}/interim/points.json")
path_sentinel2_data = Path(f"{DATA_DIR}/processed/sentinel2_data")

dataset = []

for index, row in ground_truth.iterrows():
    print(f"Riga numero: {index}")

    crop_id = row['code']
    
    path_point_data = os.path.join(path_sentinel2_data, f"point_{index}")
    
    if os.path.isdir(path_point_data):

        def estrai_media_banda(codice_banda):
            files = glob.glob(os.path.join(path_point_data, f"*{codice_banda}*.tif"))
            valori_pixel = []
            for f in files:
                try:
                    with rasterio.open(f) as src:
                        valori_pixel.append(src.read().mean())
                except:
                    pass
            if valori_pixel:
                return np.mean(valori_pixel)
            return 0

        # Ora estraiamo tutti e 6 i super-colori
        media_blu = estrai_media_banda("_B02_")
        media_verde = estrai_media_banda("_B03_")
        media_rosso = estrai_media_banda("_B04_")
        media_nir = estrai_media_banda("_B08_")
        media_swir1 = estrai_media_banda("_B11_") # Infrarosso per l'acqua 1
        media_swir2 = estrai_media_banda("_B12_") # Infrarosso per l'acqua 2

        # CALCOLO DEGLI INDICI DI VEGETAZIONE (FEATURE ENGINEERING)
        # 1. NDVI (Indice di Vigore della pianta)
        denominatore_ndvi = (media_nir + media_rosso)
        # Usiamo 'if denominatore > 0 else 0' per evitare che Python crashi dividendo per zero!
        ndvi = (media_nir - media_rosso) / denominatore_ndvi if denominatore_ndvi > 0 else 0

        # 2. NDWI (Indice di Stress Idrico / Acqua nella pianta)
        denominatore_ndwi = (media_nir + media_swir1)
        ndwi = (media_nir - media_swir1) / denominatore_ndwi if denominatore_ndwi > 0 else 0

        # Salviamo la riga per il ML solo se abbiamo trovato dei dati
        if media_blu > 0:
            dataset.append({
                'ID_Campo': index,
                'Ground_Truth': crop_id,
                'Blu_B02': media_blu,
                'Verde_B03': media_verde,
                'Rosso_B04': media_rosso,
                'NIR_B08': media_nir,
                'SWIR1_B11': media_swir1,
                'SWIR2_B12': media_swir2,
                'NDVI': ndvi,
                'NDWI': ndwi,
            })
    else:
        print(f"Cartella saltata (non trovata): {index}")
        
    # Creiamo la tabella finale
    df_finale = pd.DataFrame(dataset)
    print(f"\n✅ Estrazione completata! Tabella ML creata con {len(df_finale)} campi.")